In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries loaded!')
print('Pandas:', pd.__version__)
print('NumPy:', np.__version__)

All libraries loaded!
Pandas: 2.3.1
NumPy: 2.3.1


In [3]:
df = pd.read_csv(r'C:\Users\patel\OneDrive\Documents\Desktop\churn-analysis\data\WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('TotalCharges sample values:')
print(df['TotalCharges'].head(10).tolist())

=== Data Types ===
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

TotalCharges sample values:
['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95']


In [5]:
# Find blank TotalCharges
blank_mask = df['TotalCharges'].str.strip() == ''
print('Blank TotalCharges rows:', blank_mask.sum())
print(df[blank_mask][['customerID','tenure','MonthlyCharges','TotalCharges']])

# Fix: convert to number, blanks become NaN, fill NaN with 0
df['TotalCharges'] = df['TotalCharges'].str.strip()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('\nAfter fix:')
print('Nulls in TotalCharges:', df['TotalCharges'].isnull().sum())
print('Data type:', df['TotalCharges'].dtype)

Blank TotalCharges rows: 11
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             

After fix:
Nulls in TotalCharges: 0
Data type: float64


In [6]:
print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('Total missing:', df.isnull().sum().sum())

print('\n=== Duplicates ===')
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate customerIDs:', df['customerID'].duplicated().sum())

=== Missing Values ===
Series([], dtype: int64)
Total missing: 0

=== Duplicates ===
Duplicate rows: 0
Duplicate customerIDs: 0


In [8]:
df['Churn_Flag'] = (df['Churn'] == 'Yes').astype(int)

print('Churn_Flag value counts:')
print(df['Churn_Flag'].value_counts())
print()
print('Overall churn rate:', round(df['Churn_Flag'].mean()*100, 1), '%')

Churn_Flag value counts:
Churn_Flag
0    5174
1    1869
Name: count, dtype: int64

Overall churn rate: 26.5 %


In [9]:
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['New (0-12m)', 'Mid (12-24m)', 'Established (24-48m)', 'Loyal (48+m)'],
    include_lowest=True
)

df['ChargeTier'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 35, 65, 85, 120],
    labels=['Low (<$35)', 'Medium ($35-65)', 'High ($65-85)', 'Premium (>$85)']
)

print('Tenure Group distribution:')
print(df['TenureGroup'].value_counts().sort_index())
print()
print('Churn rate by Tenure Group:')
print(df.groupby('TenureGroup', observed=True)['Churn_Flag'].mean().mul(100).round(1))

Tenure Group distribution:
TenureGroup
New (0-12m)             2186
Mid (12-24m)            1024
Established (24-48m)    1594
Loyal (48+m)            2239
Name: count, dtype: int64

Churn rate by Tenure Group:
TenureGroup
New (0-12m)            47.40
Mid (12-24m)           28.70
Established (24-48m)   20.40
Loyal (48+m)            9.50
Name: Churn_Flag, dtype: float64


In [11]:
df.to_csv(r'C:\Users\patel\OneDrive\Documents\Desktop\churn-analysis\data\telco_clean.csv', index=False)
print('Clean file saved!')
print('Final shape:', df.shape)
print('Columns:', df.columns.tolist())

Clean file saved!
Final shape: (7043, 24)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Flag', 'TenureGroup', 'ChargeTier']
